# 00 – DQ Config (shared)
Do **not** run this on its own or schedule it. Every other DQ notebook loads it with `%run 00_dq_config`,
so table names, schemas and settings live in one place.

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime
from zoneinfo import ZoneInfo
import uuid, time

# ---- Lakehouse / schemas ----
LAKEHOUSE  = "Customer_MDM_PoC"
DQ_SCHEMA  = "dq"     # all DQ framework tables go here
SRC_SCHEMA = "dbo"    # schema where the EBS customer master tables live - CHANGE if different

DQ  = f"{LAKEHOUSE}.{DQ_SCHEMA}"
SRC = f"{LAKEHOUSE}.{SRC_SCHEMA}"   # use as {SRC}.table_name inside rule SQL

# ---- DQ tables ----
T_DIMENSION   = f"{DQ}.dq_dimension"
T_CATALOG     = f"{DQ}.dq_rule_catalog"
T_RUN         = f"{DQ}.dq_run"
T_RUN_LOG     = f"{DQ}.dq_rule_run_log"
T_FAILED_HIST = f"{DQ}.dq_failed_records_history"
T_FAILED_CURR = f"{DQ}.dq_failed_records_current"
T_DATE        = f"{DQ}.dim_date"

# ---- Operating unit (OU) tables ----
T_RUN_LOG_OU    = f"{DQ}.dq_rule_run_log_ou"   # append: one row per rule per run per OU
T_CUST_OU_MAP   = f"{DQ}.dq_customer_ou_map"   # rebuilt each run: customer -> every OU where it has a site
T_OU_REGION_MAP = f"{DQ}.dq_ou_region_map"     # maintained by you: org_id -> region / country
T_DIM_OU        = f"{DQ}.dim_operating_unit"   # rebuilt each run: OU dimension for Power BI

# Customer-level rules get their OU(s) from this mapping: ALL OUs where the account has a site
# (no primary-site choice). A customer with sites in 3 OUs counts in each of those 3 OUs.
CUSTOMER_OU_MAP_SQL = f"""
    SELECT DISTINCT CAST(cust_account_id AS STRING) AS cust_account_id,
                    CAST(org_id AS INT)             AS org_id
    FROM {SRC}.hz_cust_acct_sites_all
"""

# OU names. If this table is not in the lakehouse, the runner falls back to org_id as the name.
OU_SOURCE_SQL = f"""
    SELECT CAST(organization_id AS INT) AS org_id,
           name                         AS ou_name
    FROM {SRC}.hr_operating_units
"""

# ---- Record attributes for record-level filters (optional columns a rule's failing rows may return) ----
# If a rule's failure output (or base_sql, for condition rules) has these columns, they are stored with the
# failed records; otherwise they stay NULL.
ATTRIBUTE_COLUMNS = ["account_number", "party_number", "cust_account_id", "cust_acct_site_id"]

# ---- Run settings ----
TIMEZONE               = "Asia/Kolkata"  # Fabric Spark runs in UTC; run_date is taken in this zone
REPLACE_SAME_DAY_RUN   = True            # rerun on the same day replaces that day's results
HISTORY_RETENTION_DAYS = 180             # failed-record history older than this is deleted

# ---- Export settings (04_dq_export) ----
EXPORT_ROOT           = "/lakehouse/default/Files/dq_exports"   # default lakehouse Files mount
EXPORT_LATEST_DIR     = f"{EXPORT_ROOT}/latest"
EXPORT_ARCHIVE_DIR    = f"{EXPORT_ROOT}/archive"
EXPORT_FILE_PREFIX    = "DQ_Rule"                               # files: DQ_Rule_<id>_<rule_name>.csv
EXPORT_ARCHIVE_DAYS   = 30                                      # dated archive folders older than this are deleted

def now_local():
    return datetime.now(ZoneInfo(TIMEZONE)).replace(tzinfo=None)

print(f"DQ config loaded: DQ tables in {DQ}, source tables in {SRC}")